# Lab 5 — الانحدار من البداية للنهاية
سنستخدم المسار نفسه، لكن `y` هذه المرة رقم: متوسط قيمة السلة بالريال.

In [ ]:
import io, pandas as pd
from google.colab import files
uploaded=files.upload(); name=next(iter(uploaded))
df=pd.read_csv(io.BytesIO(uploaded[name]))
y=df['avg_basket_sar'].copy()
drop_cols=['customer_id','avg_basket_sar','churned_30d','refund_issued','support_ticket_after_snapshot','next_month_orders','signup_date','snapshot_date']
X=df.drop(columns=drop_cols,errors='ignore')
print(X.shape,y.shape)

## 1) نقسم ونبني المعالجة
لا نستخدم `stratify` لأن الهدف رقم مستمر، لكن الاختبار يبقى مغلقًا.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
num=X.select_dtypes(include='number').columns.tolist(); cat=X.select_dtypes(exclude='number').columns.tolist()
prep=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]),cat)])

## 2) DummyRegressor ثم LinearRegression
خط الأساس يتوقع الوسيط للجميع. النموذج الخطي يجب أن يقدم خطأ أقل ليضيف قيمة.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
baseline=Pipeline([('prep',prep),('model',DummyRegressor(strategy='median'))])
linear=Pipeline([('prep',prep),('model',LinearRegression())])
baseline.fit(X_train,y_train); linear.fit(X_train,y_train)
b=baseline.predict(X_test); p=linear.predict(X_test)
print('MAE baseline:',round(mean_absolute_error(y_test,b),2))
print('MAE linear:',round(mean_absolute_error(y_test,p),2))

## 3) افحص التوقع والباقي
الباقي هنا = الحقيقة − التوقع. الموجب يعني أن النموذج قلل القيمة، والسالب يعني أنه بالغ فيها.

In [ ]:
result=pd.DataFrame({'الحقيقة':y_test.iloc[:10].values,'التوقع':p[:10].round(2)})
result['الباقي']=(result['الحقيقة']-result['التوقع']).round(2)
display(result)

## التسليم
قارن MAE بين النموذجين، ثم اختر أكبر خطأ من الجدول واشرح هل بالغ النموذج أم قلل القيمة.